# Biohub Cell Tracking - SUBMISSION notebook (offline, no-internet safe)
Classical baseline: peak detection + Hungarian NN linking (thr=0.08, gate=8um -> local microJ~0.75).

**Before submitting:**
1. Attach the `zarr3-offline` wheel dataset as input (Add Input).
2. Turn **Internet OFF** in settings and Run All to VERIFY the offline zarr install works.
3. Save Version, then Submit to Competition.

In [ ]:
# Offline zarr>=3 install: auto-find the attached wheel dataset (no internet needed).
import glob, os, subprocess, sys
def find_wheeldir():
    for whl in glob.glob('/kaggle/input/**/zarr*.whl', recursive=True):
        return os.path.dirname(whl)
    return None
try:
    import zarr
    assert int(zarr.__version__.split('.')[0]) >= 3
    print('zarr already present:', zarr.__version__)
except Exception:
    wd = find_wheeldir()
    assert wd, 'No zarr wheel found under /kaggle/input -- attach the zarr3-offline dataset!'
    print('installing zarr offline from', wd)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', wd, 'zarr'])
    import zarr
    print('installed zarr:', zarr.__version__)

In [ ]:
import numpy as np, pandas as pd, time
import scipy.ndimage as ndi
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from skimage.feature import peak_local_max

# ---------------- CONFIG (tuned: thr=0.08 gate=8 -> local microJ ~0.75) ----------------
VOXEL = np.array([1.625, 0.40625, 0.40625])   # (z,y,x) um per voxel
DET = dict(sigma=(0.7, 2.0, 2.0), min_distance=3, threshold_abs=0.08,
           norm_pct=(50.0, 99.8), max_peaks=2000)
MAX_DIST_UM = 8.0

INPUT = '/kaggle/input/competitions/biohub-cell-tracking-during-development'
TEST_DIR = os.path.join(INPUT, 'test')
OUT = '/kaggle/working/submission.csv'

In [ ]:
def open_image(zarr_path):
    """Return the (T,Z,Y,X) array of an OME-Zarr sample (largest array)."""
    g = zarr.open(zarr_path, mode='r')
    if hasattr(g, 'shape'):
        return g
    best, bestsize = None, -1
    def walk(node):
        nonlocal best, bestsize
        try:
            for k in node.keys():
                sub = node[k]
                if hasattr(sub, 'shape'):
                    s = int(np.prod(sub.shape))
                    if s > bestsize: best, bestsize = sub, s
                else: walk(sub)
        except Exception: pass
    walk(g)
    return best

def detect_centroids(vol, sigma, min_distance, threshold_abs, norm_pct, max_peaks):
    v = vol.astype(np.float32)
    lo, hi = np.percentile(v, norm_pct)
    v = np.clip((v - lo) / (hi - lo + 1e-6), 0, 1)
    vs = ndi.gaussian_filter(v, sigma=sigma)
    return peak_local_max(vs, min_distance=min_distance, threshold_abs=threshold_abs,
                          exclude_border=False, num_peaks=max_peaks)

def track(arr, det=DET, max_dist_um=MAX_DIST_UM):
    T = arr.shape[0]
    nodes, edges, nid = [], [], 0
    prev_ids, prev_xyz = None, None
    for t in range(T):
        coords = detect_centroids(np.asarray(arr[t]), **det)
        ids = list(range(nid, nid + len(coords))); nid += len(coords)
        for i, c in zip(ids, coords):
            nodes.append((i, t, int(c[0]), int(c[1]), int(c[2])))
        if prev_xyz is not None and len(prev_xyz) and len(coords):
            D = cdist(prev_xyz * VOXEL, coords * VOXEL)
            r, c = linear_sum_assignment(D)
            for i, j in zip(r, c):
                if D[i, j] <= max_dist_um: edges.append((prev_ids[i], ids[j]))
        prev_ids, prev_xyz = ids, coords
    return nodes, edges

In [ ]:
# ---------------- Generate submission for ALL test datasets ----------------
test_zarrs = sorted(glob.glob(os.path.join(TEST_DIR, '*.zarr')))
print('test datasets:', len(test_zarrs))

rows = []; gid = 0; t0 = time.time()
for zp in test_zarrs:
    ds = os.path.basename(zp)[:-5]
    arr = open_image(zp)
    nodes, edges = track(arr)
    for (nid, t, z, y, x) in nodes:
        rows.append((gid, ds, 'node', nid, t, z, y, x, -1, -1)); gid += 1
    for (s, tg) in edges:
        rows.append((gid, ds, 'edge', -1, -1, -1, -1, -1, s, tg)); gid += 1
    print(f'  {ds}: {len(nodes)} nodes, {len(edges)} edges  ({time.time()-t0:.0f}s)')

cols = ['id','dataset','row_type','node_id','t','z','y','x','source_id','target_id']
sub = pd.DataFrame(rows, columns=cols)
sub.to_csv(OUT, index=False)
print('\nwrote', OUT, sub.shape)
print(sub.head(6).to_string(index=False))